# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# Week 03 – Data Contract

**Name:** Md Mainul Hoque

**Lane:** Search Intelligence

**Month:** 2026-03

---

## Objective

This notebook defines the data contract for the Search Intelligence lane, verifies the warehouse data using SQL queries, creates a small feature frame, and demonstrates feature leakage using real warehouse data.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- clicks
- impressions
- ctr
- position
- device

### Label
- future_clicks

### Context
- page
- date
- month

### Excluded
- future impressions
- future position
- future CTR
- any feature created using future information

Reason:
Future information is excluded to prevent target leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ Token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected Successfully!")
print(TABLES.keys())

Connected Successfully!
dict_keys(['dim_clients', 'dim_content', 'fact_daily', 'fact_daily_sample', 'fact_query_90d'])


In [3]:
query1 = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS cnt
FROM {TABLES['fact_daily']}
WHERE strftime(report_date,'%Y-%m')='2026-03'
GROUP BY 1,2,3
HAVING COUNT(*)>1
""").df()

query1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,cnt


Result:
No duplicate rows were found for March 2026.
Each row uniquely represents one client, one content item, and one report date.

In [4]:
query2 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE strftime(report_date,'%Y-%m')='2026-03'
""").df()

query2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


Result:
The dataset contains all records for March 2026.
The date range and total row count were verified successfully.

In [5]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [6]:
query3 = con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE strftime(report_date,'%Y-%m')='2026-03'
AND gsc_data_available IS TRUE
""").df()

query3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


Result:
Rows with gsc_data_available IS TRUE: 3,611,061.
This confirms that the required Search Console data is available for analysis.

Limitation

- Only March 2026 data is analyzed.
- Seasonal variation is not evaluated.
- Future information is excluded to avoid data leakage.
- Results may not generalize to other months.

### Limitation

- Only March 2026 data is analyzed.
- Seasonal trends are not evaluated.
- Future information is intentionally excluded.
- Results may not generalize to other months.

**Self check**

☑ Every contract answer is backed by SQL.

☑ Exactly three verification queries are included.

☑ Availability is verified using IS TRUE.

☑ Five features are described with availability explanations.

☑ Leakage demonstration is documented and removed.

☑ One limitation of the dataset is clearly stated.